In [ ]:
# ── Cell 1: Install all required packages ────────────────────────────────────
print("Installing packages...")

import subprocess, sys

packages = [
    "flask",
    "flask-cors",
    "pyngrok",
    "mediapipe",
    "numpy",
    "Pillow",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("✅ All packages installed.")

Installing packages...
✅ All packages installed.


In [ ]:
# ── Cell 2: Write index.html — the complete pose correction web app ───────────

import os
os.makedirs("/content/pose_app/static", exist_ok=True)

HTML = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0, user-scalable=no">
<title>PoseAI — Real-Time Form Correction</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Syne:wght@400;600;700;800&family=DM+Sans:ital,opsz,wght@0,9..40,300;0,9..40,400;0,9..40,500;1,9..40,300&display=swap" rel="stylesheet">
<style>
:root {
  --bg: #F7F4EF;
  --white: #FFFFFF;
  --ink: #1A1714;
  --ink-soft: #4A4540;
  --ink-muted: #9A9490;
  --sage: #3D7A52;
  --sage-light: #E5EFE8;
  --coral: #E8604C;
  --gold: #C9963A;
  --border: #E2DDD7;
  --panel: rgba(255,255,255,0.92);
}
*, *::before, *::after { box-sizing: border-box; margin:0; padding:0; }
body {
  font-family: 'DM Sans', sans-serif;
  background: var(--bg);
  color: var(--ink);
  height: 100vh;
  overflow: hidden;
  display: flex;
  flex-direction: column;
}

/* ── TOP NAV ── */
nav {
  height: 52px;
  background: var(--white);
  border-bottom: 1px solid var(--border);
  display: flex;
  align-items: center;
  justify-content: space-between;
  padding: 0 20px;
  flex-shrink: 0;
  z-index: 10;
}
.nav-logo {
  font-family: 'Syne', sans-serif;
  font-weight: 800;
  font-size: 1.1rem;
  color: var(--ink);
  display: flex; align-items: center; gap: 8px;
}
.live-dot {
  width: 7px; height: 7px;
  background: var(--sage);
  border-radius: 50%;
  animation: blink 1.5s ease-in-out infinite;
}
@keyframes blink { 0%,100%{opacity:1} 50%{opacity:0.3} }
.nav-right { display: flex; align-items: center; gap: 10px; }
.nav-badge {
  font-size: 0.7rem; font-weight: 600;
  color: var(--sage);
  background: var(--sage-light);
  border: 1px solid #B8D4BC;
  padding: 3px 10px; border-radius: 100px;
  letter-spacing: 0.05em;
}
#fps-display {
  font-size: 0.72rem; color: var(--ink-muted);
  font-family: 'DM Sans', monospace;
}

/* ── MAIN LAYOUT ── */
.main {
  flex: 1;
  display: grid;
  grid-template-columns: 1fr 300px;
  gap: 0;
  overflow: hidden;
}

/* ── CAMERA AREA ── */
.camera-area {
  position: relative;
  background: #111;
  overflow: hidden;
}
#video {
  position: absolute; inset: 0;
  width: 100%; height: 100%;
  object-fit: cover;
}
#overlay-canvas {
  position: absolute; inset: 0;
  width: 100%; height: 100%;
  object-fit: cover;
  pointer-events: none;
}

/* Scan line effect while loading */
.scan-line {
  position: absolute;
  left: 0; right: 0;
  height: 2px;
  background: linear-gradient(to right, transparent, var(--sage), transparent);
  animation: scan 2s linear infinite;
  display: none;
}
@keyframes scan {
  0% { top: 0; opacity: 0; }
  10% { opacity: 1; }
  90% { opacity: 1; }
  100% { top: 100%; opacity: 0; }
}

/* Camera HUD overlays */
.hud-topleft {
  position: absolute; top: 14px; left: 14px;
  display: flex; flex-direction: column; gap: 8px;
}
.hud-tag {
  background: rgba(0,0,0,0.55);
  backdrop-filter: blur(8px);
  border: 1px solid rgba(255,255,255,0.12);
  border-radius: 8px;
  padding: 6px 12px;
  font-size: 0.72rem;
  font-weight: 600;
  color: #fff;
  letter-spacing: 0.06em;
  text-transform: uppercase;
}
.hud-tag.green { border-color: rgba(61,122,82,0.5); color: #6ddf94; }
.hud-tag.yellow { border-color: rgba(201,150,58,0.5); color: #f5c55a; }
.hud-tag.red { border-color: rgba(232,96,76,0.5); color: #f08070; }

#no-pose-msg {
  position: absolute; inset: 0;
  display: flex; flex-direction: column;
  align-items: center; justify-content: center;
  background: rgba(17,17,17,0.7);
  backdrop-filter: blur(4px);
  color: rgba(255,255,255,0.6);
  font-size: 0.9rem;
  gap: 14px;
}
#no-pose-msg svg { opacity: 0.4; }

/* ── RIGHT PANEL ── */
.right-panel {
  background: var(--white);
  border-left: 1px solid var(--border);
  display: flex;
  flex-direction: column;
  overflow-y: auto;
  overflow-x: hidden;
}

/* ── SCORE BLOCK ── */
.score-block {
  padding: 20px;
  border-bottom: 1px solid var(--border);
  text-align: center;
  background: var(--white);
}
.score-ring-wrap { position: relative; width: 120px; height: 120px; margin: 0 auto 12px; }
.score-ring-wrap svg { transform: rotate(-90deg); }
#score-ring-track { stroke: var(--border); }
#score-ring-fill {
  stroke: var(--sage);
  stroke-dasharray: 283;
  stroke-dashoffset: 283;
  transition: stroke-dashoffset 0.6s ease, stroke 0.3s ease;
}
.score-center {
  position: absolute; inset: 0;
  display: flex; flex-direction: column;
  align-items: center; justify-content: center;
}
#score-number {
  font-family: 'Syne', sans-serif;
  font-size: 2rem;
  font-weight: 800;
  color: var(--ink);
  line-height: 1;
}
#score-label {
  font-size: 0.6rem;
  color: var(--ink-muted);
  letter-spacing: 0.1em;
  text-transform: uppercase;
  margin-top: 2px;
}
#status-pill {
  display: inline-block;
  padding: 4px 14px;
  border-radius: 100px;
  font-size: 0.72rem;
  font-weight: 600;
  letter-spacing: 0.08em;
  background: var(--sage-light);
  color: var(--sage);
  border: 1px solid #B8D4BC;
  transition: all 0.3s ease;
}
#status-pill.minor { background: #FDF6E8; color: var(--gold); border-color: #E8D4A0; }
#status-pill.major { background: #FDF0EE; color: var(--coral); border-color: #F0C0B8; }

/* ── EXERCISE SELECT ── */
.ex-block {
  padding: 16px 20px;
  border-bottom: 1px solid var(--border);
}
.block-label {
  font-size: 0.68rem;
  font-weight: 600;
  color: var(--ink-muted);
  text-transform: uppercase;
  letter-spacing: 0.1em;
  margin-bottom: 8px;
}
#exercise-select {
  width: 100%;
  padding: 9px 12px;
  border-radius: 8px;
  border: 1px solid var(--border);
  background: var(--bg);
  color: var(--ink);
  font-family: 'DM Sans', sans-serif;
  font-size: 0.88rem;
  outline: none;
  cursor: pointer;
}
.mode-row { display: flex; gap: 8px; margin-top: 10px; }
.mode-btn {
  flex: 1;
  padding: 8px;
  border-radius: 8px;
  border: 1px solid var(--border);
  background: var(--white);
  font-family: 'DM Sans', sans-serif;
  font-size: 0.78rem;
  font-weight: 500;
  color: var(--ink-soft);
  cursor: pointer;
  transition: all 0.2s ease;
  text-align: center;
}
.mode-btn.active {
  background: var(--ink);
  color: #fff;
  border-color: var(--ink);
}

/* ── REPS ── */
.reps-block {
  padding: 16px 20px;
  border-bottom: 1px solid var(--border);
  display: flex;
  align-items: center;
  justify-content: space-between;
}
.reps-left { display: flex; flex-direction: column; }
.reps-num {
  font-family: 'Syne', sans-serif;
  font-size: 2.2rem;
  font-weight: 800;
  color: var(--ink);
  line-height: 1;
}
.reps-label { font-size: 0.68rem; color: var(--ink-muted); text-transform: uppercase; letter-spacing: 0.1em; }
#btn-reset {
  padding: 8px 14px;
  border-radius: 8px;
  border: 1px solid var(--border);
  background: var(--white);
  font-family: 'DM Sans', sans-serif;
  font-size: 0.78rem;
  color: var(--ink-soft);
  cursor: pointer;
  transition: all 0.2s ease;
}
#btn-reset:hover { background: var(--bg); }

/* ── CORRECTIONS ── */
.corrections-block {
  padding: 16px 20px;
  border-bottom: 1px solid var(--border);
  flex: 1;
}
#correction-list { margin-top: 8px; display: flex; flex-direction: column; gap: 8px; }
.correction-item {
  display: flex; align-items: flex-start; gap: 8px;
  background: #FDF0EE;
  border: 1px solid #F0C0B8;
  border-radius: 8px;
  padding: 8px 10px;
  font-size: 0.82rem;
  color: var(--ink-soft);
  line-height: 1.4;
  animation: slide-in 0.3s ease;
}
@keyframes slide-in { from { opacity:0; transform:translateX(8px); } to { opacity:1; transform:translateX(0); } }
.correction-dot { color: var(--coral); font-size: 0.9rem; margin-top: 1px; flex-shrink: 0; }
.good-form {
  background: var(--sage-light);
  border: 1px solid #B8D4BC;
  border-radius: 8px;
  padding: 10px;
  font-size: 0.82rem;
  color: var(--sage);
  text-align: center;
  font-weight: 500;
}

/* ── JOINT SCORES ── */
.joints-block {
  padding: 16px 20px;
  border-bottom: 1px solid var(--border);
}
#joints-list { margin-top: 8px; display: flex; flex-direction: column; gap: 6px; }
.joint-row { display: flex; flex-direction: column; gap: 3px; }
.joint-row-top { display: flex; justify-content: space-between; align-items: center; }
.joint-name { font-size: 0.72rem; color: var(--ink-soft); text-transform: capitalize; }
.joint-val { font-size: 0.72rem; font-weight: 600; color: var(--ink); }
.joint-bar-bg { height: 4px; background: var(--border); border-radius: 2px; overflow: hidden; }
.joint-bar-fill { height: 100%; border-radius: 2px; transition: width 0.4s ease, background 0.3s ease; }

/* ── CAMERA CONTROLS ── */
.controls-block {
  padding: 16px 20px;
}
.controls-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 8px; }
.ctrl-btn {
  padding: 10px 8px;
  border-radius: 10px;
  border: 1px solid var(--border);
  background: var(--white);
  font-family: 'DM Sans', sans-serif;
  font-size: 0.78rem;
  color: var(--ink-soft);
  cursor: pointer;
  transition: all 0.2s ease;
  display: flex; align-items: center; justify-content: center; gap: 6px;
}
.ctrl-btn:hover { background: var(--bg); color: var(--ink); }
.ctrl-btn.active { background: var(--sage); color: #fff; border-color: var(--sage); }

/* ── SLIDERS ── */
.sliders-block {
  padding: 0 20px 20px;
}
.slider-row { margin-bottom: 12px; }
.slider-label-row {
  display: flex; justify-content: space-between; align-items: center;
  margin-bottom: 4px;
}
.slider-name { font-size: 0.72rem; color: var(--ink-muted); }
.slider-val { font-size: 0.72rem; font-weight: 600; color: var(--ink); }
input[type="range"] {
  width: 100%;
  -webkit-appearance: none;
  height: 4px;
  background: var(--border);
  border-radius: 2px;
  outline: none;
}
input[type="range"]::-webkit-slider-thumb {
  -webkit-appearance: none;
  width: 14px; height: 14px;
  border-radius: 50%;
  background: var(--ink);
  cursor: pointer;
  transition: background 0.2s;
}
input[type="range"]::-webkit-slider-thumb:hover { background: var(--sage); }

/* ── LOADING OVERLAY ── */
#loading-overlay {
  position: fixed; inset: 0;
  background: var(--bg);
  z-index: 100;
  display: flex; flex-direction: column;
  align-items: center; justify-content: center;
  gap: 16px;
  transition: opacity 0.5s ease;
}
.spinner {
  width: 44px; height: 44px;
  border: 3px solid var(--border);
  border-top: 3px solid var(--sage);
  border-radius: 50%;
  animation: spin 0.8s linear infinite;
}
@keyframes spin { to { transform: rotate(360deg); } }
#load-msg {
  font-size: 0.9rem;
  color: var(--ink-soft);
  font-weight: 300;
}
.load-title {
  font-family: 'Syne', sans-serif;
  font-size: 1.5rem;
  font-weight: 800;
  color: var(--ink);
  letter-spacing: -0.02em;
}

/* ── RESPONSIVE ── */
@media (max-width: 700px) {
  .main { grid-template-columns: 1fr; grid-template-rows: 55vh 1fr; }
  .right-panel { border-left: none; border-top: 1px solid var(--border); }
}
</style>
</head>
<body>

<!-- Loading -->
<div id="loading-overlay">
  <div class="spinner"></div>
  <div class="load-title">PoseAI</div>
  <div id="load-msg">Loading MediaPipe models...</div>
</div>

<!-- Nav -->
<nav>
  <div class="nav-logo">
    <div class="live-dot"></div>
    PoseAI
  </div>
  <div class="nav-right">
    <span id="fps-display">-- fps</span>
    <div class="nav-badge">LIVE</div>
  </div>
</nav>

<!-- Main -->
<div class="main">

  <!-- Camera -->
  <div class="camera-area">
    <video id="video" autoplay playsinline muted></video>
    <canvas id="overlay-canvas"></canvas>
    <div class="scan-line" id="scan-line"></div>

    <!-- HUD overlays on camera -->
    <div class="hud-topleft">
      <div class="hud-tag" id="pose-name-tag">Detecting...</div>
      <div class="hud-tag green" id="form-status-tag">READY</div>
    </div>

    <!-- No pose message -->
    <div id="no-pose-msg">
      <svg width="48" height="48" viewBox="0 0 24 24" fill="none" stroke="white" stroke-width="1.5"><circle cx="12" cy="5" r="2"/><path d="M12 7v6l3 3"/><path d="M8.5 8.5C7 10 6.5 11.5 7 13l2 3h6l2-3c.5-1.5 0-3-1.5-4.5"/></svg>
      <span>Stand in front of the camera</span>
    </div>
  </div>

  <!-- Right Panel -->
  <div class="right-panel">

    <!-- Form Score -->
    <div class="score-block">
      <div class="score-ring-wrap">
        <svg width="120" height="120" viewBox="0 0 120 120">
          <circle id="score-ring-track" cx="60" cy="60" r="45" fill="none" stroke-width="8"/>
          <circle id="score-ring-fill" cx="60" cy="60" r="45" fill="none" stroke-width="8" stroke-linecap="round"/>
        </svg>
        <div class="score-center">
          <div id="score-number">0</div>
          <div id="score-label">SCORE</div>
        </div>
      </div>
      <span id="status-pill">READY</span>
    </div>

    <!-- Exercise Selector -->
    <div class="ex-block">
      <div class="block-label">Exercise / Pose</div>
      <select id="exercise-select" onchange="onExerciseChange()">
        <option value="">— Auto Detect —</option>
        <optgroup label="🧘 Yoga Asanas">
          <option value="tadasana">Mountain Pose (Tadasana)</option>
          <option value="vrukshasana">Tree Pose (Vrukshasana)</option>
          <option value="warrior_1">Warrior I</option>
          <option value="warrior_2">Warrior II</option>
          <option value="trikonasana">Triangle Pose</option>
          <option value="ardhachandrasana">Half Moon</option>
          <option value="downward_dog">Downward Dog</option>
          <option value="balasana">Child's Pose</option>
          <option value="bhujangasana">Cobra Pose</option>
          <option value="plank">Plank Pose</option>
          <option value="paschimottanasana">Seated Forward Bend</option>
          <option value="navasana">Boat Pose</option>
          <option value="setu_bandhasana">Bridge Pose</option>
          <option value="pigeon_pose">Pigeon Pose</option>
          <option value="utkatasana">Chair Pose</option>
        </optgroup>
        <optgroup label="💪 Fitness Exercises">
          <option value="squat">Squat</option>
          <option value="pushup">Push-Up</option>
          <option value="deadlift">Deadlift</option>
          <option value="bicep_curl">Bicep Curl</option>
          <option value="shoulder_press">Shoulder Press</option>
          <option value="lunge">Lunge</option>
          <option value="plank_fitness">Plank Hold</option>
          <option value="hip_thrust">Hip Thrust / Glute Bridge</option>
          <option value="jumping_jacks">Jumping Jacks</option>
        </optgroup>
      </select>
      <div class="mode-row">
        <button class="mode-btn active" id="btn-auto" onclick="setMode('auto')">🔍 Auto</button>
        <button class="mode-btn" id="btn-manual" onclick="setMode('manual')">🔒 Manual</button>
      </div>
    </div>

    <!-- Reps -->
    <div class="reps-block">
      <div class="reps-left">
        <div class="reps-num" id="rep-num">0</div>
        <div class="reps-label">Reps</div>
      </div>
      <button id="btn-reset" onclick="resetReps()">↺ Reset</button>
    </div>

    <!-- Corrections -->
    <div class="corrections-block">
      <div class="block-label">Form Feedback</div>
      <div id="correction-list">
        <div class="good-form">✓ Position yourself in frame</div>
      </div>
    </div>

    <!-- Joint Scores -->
    <div class="joints-block">
      <div class="block-label">Joint Scores</div>
      <div id="joints-list"></div>
    </div>

    <!-- Camera Controls -->
    <div class="controls-block">
      <div class="block-label">Camera</div>
      <div class="controls-grid">
        <button class="ctrl-btn" onclick="flipCamera()">
          <svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round"><path d="M21 7v6h-6"/><path d="M3 17v-6h6"/><path d="M18.49 4.01A10 10 0 0 0 2.5 13.5"/><path d="M5.51 19.99A10 10 0 0 0 21.5 10.5"/></svg>
          Flip
        </button>
        <button class="ctrl-btn" id="btn-mirror" onclick="toggleMirror()">
          <svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2" stroke-linecap="round"><line x1="12" y1="2" x2="12" y2="22"/><path d="M8 6l-4 4 4 4"/><path d="M16 6l4 4-4 4"/></svg>
          Mirror
        </button>
        <button class="ctrl-btn" id="btn-skeleton" onclick="toggleSkeleton()">
          <svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><circle cx="12" cy="4" r="2"/><path d="M12 6v6l3 3M12 12l-3 3"/><path d="M9 20h6"/></svg>
          Skeleton
        </button>
        <button class="ctrl-btn" id="btn-pause" onclick="togglePause()">
          <svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><rect x="6" y="4" width="4" height="16"/><rect x="14" y="4" width="4" height="16"/></svg>
          Pause
        </button>
      </div>
    </div>

    <!-- Sliders -->
    <div class="sliders-block">
      <div class="block-label" style="margin-bottom:10px">Adjustments</div>
      <div class="slider-row">
        <div class="slider-label-row">
          <span class="slider-name">Brightness</span>
          <span class="slider-val" id="val-brightness">100%</span>
        </div>
        <input type="range" id="sl-brightness" min="50" max="200" value="100" oninput="applyFilters()">
      </div>
      <div class="slider-row">
        <div class="slider-label-row">
          <span class="slider-name">Contrast</span>
          <span class="slider-val" id="val-contrast">100%</span>
        </div>
        <input type="range" id="sl-contrast" min="50" max="200" value="100" oninput="applyFilters()">
      </div>
      <div class="slider-row">
        <div class="slider-label-row">
          <span class="slider-name">Saturation</span>
          <span class="slider-val" id="val-saturation">100%</span>
        </div>
        <input type="range" id="sl-saturation" min="0" max="200" value="100" oninput="applyFilters()">
      </div>
      <div class="slider-row">
        <div class="slider-label-row">
          <span class="slider-name">Detection Sensitivity</span>
          <span class="slider-val" id="val-sensitivity">60%</span>
        </div>
        <input type="range" id="sl-sensitivity" min="30" max="90" value="60" oninput="updateSensitivity()">
      </div>
    </div>

  </div><!-- end right-panel -->
</div><!-- end main -->

<script type="module">
// ═══════════════════════════════════════════════════════════
//  EXERCISE DEFINITIONS — 30 poses
// ═══════════════════════════════════════════════════════════
const EXERCISES = {
  tadasana:{name:"Mountain Pose",cat:"yoga",refAngles:{left_knee:175,right_knee:175,left_hip:175,right_hip:175,left_elbow:170,right_elbow:170,spine_upper:90},corrections:{left_knee:"Straighten left knee",right_knee:"Straighten right knee",left_hip:"Stand tall, extend hips",spine_upper:"Keep shoulders level"},repJoints:null,repRange:null},
  vrukshasana:{name:"Tree Pose",cat:"yoga",refAngles:{left_knee:170,right_knee:70,left_hip:170,right_hip:120,left_shoulder:150,right_shoulder:150,spine_upper:90},corrections:{left_knee:"Keep standing leg straight",right_knee:"Bend raised knee outward more",left_shoulder:"Raise arms overhead",spine_upper:"Don't lean sideways"},repJoints:null,repRange:null},
  warrior_1:{name:"Warrior I",cat:"yoga",refAngles:{left_knee:90,right_knee:170,left_hip:100,right_hip:160,left_shoulder:170,right_shoulder:170,left_elbow:175,right_elbow:175,spine_upper:80},corrections:{left_knee:"Bend front knee to 90°",right_knee:"Keep back leg straight",left_shoulder:"Reach arms straight up",spine_upper:"Square hips forward"},repJoints:null,repRange:null},
  warrior_2:{name:"Warrior II",cat:"yoga",refAngles:{left_knee:90,right_knee:170,left_hip:100,right_hip:130,left_shoulder:90,right_shoulder:90,left_elbow:175,right_elbow:175,spine_upper:90},corrections:{left_knee:"Bend front knee over ankle",right_knee:"Straighten back leg",left_shoulder:"Extend arms parallel to floor",spine_upper:"Open hips to the side"},repJoints:null,repRange:null},
  trikonasana:{name:"Triangle Pose",cat:"yoga",refAngles:{left_knee:175,right_knee:175,left_hip:120,right_hip:135,left_shoulder:90,right_shoulder:90,spine_upper:75},corrections:{left_knee:"Keep both legs straight",left_hip:"Tilt torso sideways more",left_shoulder:"Extend arms open",spine_upper:"Open chest to ceiling"},repJoints:null,repRange:null},
  ardhachandrasana:{name:"Half Moon",cat:"yoga",refAngles:{left_knee:175,right_knee:80,left_hip:90,right_hip:175,left_shoulder:90,right_shoulder:170,spine_upper:90},corrections:{right_knee:"Balance leg nearly straight",left_hip:"Raise lifted leg parallel",left_shoulder:"Extend top arm up",spine_upper:"Stack hips vertically"},repJoints:null,repRange:null},
  downward_dog:{name:"Downward Dog",cat:"yoga",refAngles:{left_knee:170,right_knee:170,left_hip:60,right_hip:60,left_shoulder:175,right_shoulder:175,left_elbow:175,right_elbow:175,spine_upper:50},corrections:{left_knee:"Straighten knees, lengthen hamstrings",left_hip:"Push hips higher to the ceiling",left_shoulder:"Press hands down, straighten arms",spine_upper:"Create long line from hands to hips"},repJoints:null,repRange:null},
  balasana:{name:"Child's Pose",cat:"yoga",refAngles:{left_knee:30,right_knee:30,left_hip:30,right_hip:30,left_shoulder:160,right_shoulder:160,left_elbow:175,right_elbow:175,spine_upper:50},corrections:{left_knee:"Sink hips further to heels",left_hip:"Let hips fall onto heels",left_shoulder:"Stretch arms further forward",spine_upper:"Relax — let back round naturally"},repJoints:null,repRange:null},
  bhujangasana:{name:"Cobra Pose",cat:"yoga",refAngles:{left_elbow:100,right_elbow:100,left_shoulder:100,right_shoulder:100,left_hip:160,right_hip:160,left_knee:170,right_knee:170,spine_upper:60},corrections:{left_elbow:"Bend elbows slightly",left_hip:"Keep hips pressed to mat",left_knee:"Keep legs flat and straight",spine_upper:"Lift chest higher, open your heart"},repJoints:null,repRange:null},
  plank:{name:"Plank Pose",cat:"yoga",refAngles:{left_elbow:175,right_elbow:175,left_shoulder:90,right_shoulder:90,left_hip:175,right_hip:175,left_knee:175,right_knee:175,spine_upper:90},corrections:{left_hip:"Keep hips level — don't sag",right_hip:"Keep hips level",left_elbow:"Straighten arms fully",spine_upper:"Maintain rigid straight line"},repJoints:null,repRange:null},
  paschimottanasana:{name:"Seated Forward Bend",cat:"yoga",refAngles:{left_hip:45,right_hip:45,left_knee:175,right_knee:175,left_shoulder:100,right_shoulder:100,spine_upper:45},corrections:{left_knee:"Keep legs flat and straight",left_hip:"Hinge deeper at hips, reach forward",spine_upper:"Elongate spine as you fold"},repJoints:null,repRange:null},
  navasana:{name:"Boat Pose",cat:"yoga",refAngles:{left_knee:150,right_knee:150,left_hip:60,right_hip:60,left_elbow:175,right_elbow:175,left_shoulder:70,right_shoulder:70,spine_upper:70},corrections:{left_knee:"Straighten legs to raise higher",left_hip:"Balance on sit bones",spine_upper:"Avoid rounding — keep spine long"},repJoints:null,repRange:null},
  setu_bandhasana:{name:"Bridge Pose",cat:"yoga",refAngles:{left_knee:90,right_knee:90,left_hip:130,right_hip:130,left_shoulder:90,right_shoulder:90,spine_upper:45},corrections:{left_knee:"Feet flat, shins vertical",left_hip:"Drive hips higher, squeeze glutes",spine_upper:"Keep upper back flat on mat"},repJoints:null,repRange:null},
  pigeon_pose:{name:"Pigeon Pose",cat:"yoga",refAngles:{left_knee:70,right_knee:170,left_hip:80,right_hip:140,left_shoulder:90,right_shoulder:90,spine_upper:60},corrections:{right_knee:"Extend back leg fully behind you",left_hip:"Square hips toward the front",spine_upper:"Keep torso upright or fold gently"},repJoints:null,repRange:null},
  utkatasana:{name:"Chair Pose",cat:"yoga",refAngles:{left_knee:100,right_knee:100,left_hip:100,right_hip:100,left_shoulder:155,right_shoulder:155,left_elbow:175,right_elbow:175,spine_upper:80},corrections:{left_knee:"Sink hips lower — like sitting in a chair",left_shoulder:"Raise arms straight overhead",spine_upper:"Lift chest, avoid excessive lean"},repJoints:null,repRange:null},
  squat:{name:"Squat",cat:"fitness",refAngles:{left_knee:90,right_knee:90,left_hip:80,right_hip:80,left_shoulder:80,right_shoulder:80,spine_upper:80,spine_lower:90},corrections:{left_knee:"Squat deeper — thighs to parallel",right_knee:"Keep right knee aligned with toes",left_hip:"Sit back more into the squat",spine_upper:"Keep chest up, don't lean forward"},repJoints:["left_knee","right_knee"],repRange:[90,170]},
  pushup:{name:"Push-Up",cat:"fitness",refAngles:{left_elbow:90,right_elbow:90,left_shoulder:45,right_shoulder:45,left_hip:175,right_hip:175,left_knee:175,right_knee:175,spine_upper:90},corrections:{left_elbow:"Lower chest to 90° at elbow",right_elbow:"Keep elbows close to body",left_hip:"Don't sag hips — keep body rigid",spine_upper:"Maintain straight plank alignment"},repJoints:["left_elbow","right_elbow"],repRange:[90,170]},
  deadlift:{name:"Deadlift",cat:"fitness",refAngles:{left_knee:160,right_knee:160,left_hip:70,right_hip:70,left_shoulder:80,right_shoulder:80,spine_upper:80,spine_lower:85,left_elbow:175,right_elbow:175},corrections:{left_hip:"Hinge at hips — push them back",spine_upper:"Flat back — no rounding!",spine_lower:"Brace core, neutral lower back",left_knee:"Slight knee bend only"},repJoints:["left_hip","right_hip"],repRange:[70,175]},
  bicep_curl:{name:"Bicep Curl",cat:"fitness",refAngles:{left_elbow:40,right_elbow:40,left_shoulder:20,right_shoulder:20,spine_upper:90,left_hip:175,right_hip:175},corrections:{left_elbow:"Curl higher toward shoulder",right_elbow:"Curl right arm higher",left_shoulder:"Keep upper arm still — no swinging",spine_upper:"Don't lean back"},repJoints:["left_elbow","right_elbow"],repRange:[40,170]},
  shoulder_press:{name:"Shoulder Press",cat:"fitness",refAngles:{left_elbow:170,right_elbow:170,left_shoulder:165,right_shoulder:165,spine_upper:85,spine_lower:90,left_hip:170,right_hip:170},corrections:{left_elbow:"Press fully overhead — extend arms",left_shoulder:"Keep wrists directly over elbows",spine_upper:"Brace core — don't arch lower back"},repJoints:["left_elbow","right_elbow"],repRange:[90,170]},
  lunge:{name:"Lunge",cat:"fitness",refAngles:{left_knee:90,right_knee:90,left_hip:100,right_hip:160,spine_upper:85,left_shoulder:10,right_shoulder:10},corrections:{left_knee:"Front knee at 90° over ankle",right_knee:"Lower back knee toward floor",spine_upper:"Keep torso upright",left_hip:"Don't lean forward"},repJoints:["left_knee","right_knee"],repRange:[90,170]},
  plank_fitness:{name:"Plank Hold",cat:"fitness",refAngles:{left_elbow:90,right_elbow:90,left_hip:175,right_hip:175,left_knee:175,right_knee:175,spine_upper:90},corrections:{left_hip:"Keep hips perfectly level",right_hip:"Don't let hips drop or pike",left_elbow:"Elbows directly under shoulders",spine_upper:"Rigid body from head to heels"},repJoints:null,repRange:null},
  hip_thrust:{name:"Hip Thrust",cat:"fitness",refAngles:{left_knee:90,right_knee:90,left_hip:160,right_hip:160,spine_upper:90,spine_lower:90},corrections:{left_hip:"Drive hips higher, squeeze glutes",right_hip:"Push right hip up equally",left_knee:"Feet flat, shins vertical",spine_upper:"Don't hyperextend at the top"},repJoints:["left_hip","right_hip"],repRange:[90,160]},
  jumping_jacks:{name:"Jumping Jacks",cat:"fitness",refAngles:{left_shoulder:160,right_shoulder:160,left_knee:175,right_knee:175,left_hip:155,right_hip:155,left_elbow:170,right_elbow:170},corrections:{left_shoulder:"Raise arms fully overhead",right_shoulder:"Extend right arm fully",left_hip:"Jump feet wider apart"},repJoints:["left_shoulder","right_shoulder"],repRange:[20,160]},
};

const CLASS_NAMES = Object.keys(EXERCISES);

// ═══════════════════════════════════════════════════════════
//  STATE
// ═══════════════════════════════════════════════════════════
let autoDetect = true;
let exerciseKey = 'tadasana';
let repCount = 0;
let repState = 'idle';
let lastRepTime = 0;
let paused = false;
let mirrorMode = true;
let showSkeleton = true;
let poseLandmarker = null;
let lastVideoTime = -1;
let poseTimestamp = 0;
let facingMode = 'user';
let detectionConf = 0.6;
let poseHistory = [];
const HISTORY = 40;

// FPS tracking
let frames = 0, fpsTimer = Date.now();

// ═══════════════════════════════════════════════════════════
//  ANGLE MATH
// ═══════════════════════════════════════════════════════════
function angleBetween(a, v, b) {
  const v1 = [a.x-v.x, a.y-v.y, (a.z||0)-(v.z||0)];
  const v2 = [b.x-v.x, b.y-v.y, (b.z||0)-(v.z||0)];
  const n1 = Math.sqrt(v1[0]**2+v1[1]**2+v1[2]**2);
  const n2 = Math.sqrt(v2[0]**2+v2[1]**2+v2[2]**2);
  if (n1<1e-6||n2<1e-6) return 0;
  const cos = (v1[0]*v2[0]+v1[1]*v2[1]+v1[2]*v2[2])/(n1*n2);
  return Math.acos(Math.max(-1,Math.min(1,cos)))*180/Math.PI;
}

function getAllAngles(lm) {
  return {
    left_elbow:     angleBetween(lm[11],lm[13],lm[15]),
    right_elbow:    angleBetween(lm[12],lm[14],lm[16]),
    left_shoulder:  angleBetween(lm[13],lm[11],lm[23]),
    right_shoulder: angleBetween(lm[14],lm[12],lm[24]),
    left_hip:       angleBetween(lm[11],lm[23],lm[25]),
    right_hip:      angleBetween(lm[12],lm[24],lm[26]),
    left_knee:      angleBetween(lm[23],lm[25],lm[27]),
    right_knee:     angleBetween(lm[24],lm[26],lm[28]),
    left_ankle:     angleBetween(lm[25],lm[27],lm[31]),
    right_ankle:    angleBetween(lm[26],lm[28],lm[32]),
    left_hip_side:  angleBetween(lm[25],lm[23],lm[24]),
    right_hip_side: angleBetween(lm[26],lm[24],lm[23]),
    spine_upper:    angleBetween(lm[11],lm[12],lm[24]),
    spine_lower:    angleBetween(lm[23],lm[24],lm[26]),
  };
}

// ═══════════════════════════════════════════════════════════
//  RULE-BASED CLASSIFIER
// ═══════════════════════════════════════════════════════════
function classifyPose(angles) {
  let best = exerciseKey, bestScore = -Infinity;
  for (const [key, ex] of Object.entries(EXERCISES)) {
    let score = 0, w = 0;
    for (const [j, ref] of Object.entries(ex.refAngles)) {
      if (angles[j]===undefined) continue;
      const dev = Math.abs(angles[j]-ref);
      score += 100*Math.exp(-(dev**2)/800);
      w++;
    }
    if (w>0) score /= w;
    if (key===exerciseKey) score += 12; // hysteresis
    if (score > bestScore) { bestScore = score; best = key; }
  }
  // Temporal smoothing
  poseHistory.push(best);
  if (poseHistory.length > HISTORY) poseHistory.shift();
  const counts = {};
  let winner = exerciseKey, max = 0;
  for (const p of poseHistory) {
    counts[p] = (counts[p]||0)+1;
    if (counts[p]>max) { max=counts[p]; winner=p; }
  }
  return max > HISTORY*0.75 ? winner : exerciseKey;
}

// ═══════════════════════════════════════════════════════════
//  FORM EVALUATOR
// ═══════════════════════════════════════════════════════════
function evaluateForm(angles, key) {
  const ex = EXERCISES[key];
  if (!ex) return {score:0,corrections:[],jointScores:{}};
  let total = 0, cnt = 0;
  const corrections = [], jointScores = {};

  for (const [joint, ref] of Object.entries(ex.refAngles)) {
    if (angles[joint]===undefined) continue;
    const dev = Math.abs(angles[joint]-ref);
    const thr = 12;
    const norm = Math.min(dev/(thr*3), 1);
    const js = Math.max(0, 100*(1-norm));
    jointScores[joint] = js;
    total += js; cnt++;
    if (dev > 6 && ex.corrections[joint]) {
      corrections.push([joint, dev, ex.corrections[joint]]);
    }
  }
  corrections.sort((a,b)=>b[1]-a[1]);
  const score = cnt > 0 ? total/cnt : 0;
  const status = score>=85?'correct':score>=65?'minor':'major';
  return {score:Math.round(score), corrections:corrections.slice(0,3), status, jointScores};
}

// ═══════════════════════════════════════════════════════════
//  REP COUNTER
// ═══════════════════════════════════════════════════════════
function updateReps(key, angles) {
  const ex = EXERCISES[key];
  if (!ex?.repJoints || !ex?.repRange) return;
  const vals = ex.repJoints.map(j=>angles[j]).filter(v=>v!==undefined);
  if (!vals.length) return;
  const avg = vals.reduce((a,b)=>a+b,0)/vals.length;
  const lo = Math.min(...ex.repRange), hi = Math.max(...ex.repRange);
  const mid = (lo+hi)/2;
  const now = Date.now()/1000;

  if (repState==='idle') { if (avg>hi-15) repState='top'; else if (avg<lo+15) repState='bottom'; }
  else if (repState==='top') { if (avg<mid) repState='descending'; }
  else if (repState==='descending') { if (avg<lo+15) repState='bottom'; }
  else if (repState==='bottom') { if (avg>mid) repState='ascending'; }
  else if (repState==='ascending') {
    if (avg>hi-15) {
      if (now-lastRepTime>0.8) { repCount++; lastRepTime=now; }
      repState='top';
    }
  }
}

// ═══════════════════════════════════════════════════════════
//  SKELETON DRAWING
// ═══════════════════════════════════════════════════════════
const CONNECTIONS = [
  [11,13],[13,15],[12,14],[14,16],[11,12],[23,24],[11,23],[12,24],
  [23,25],[25,27],[27,29],[27,31],[24,26],[26,28],[28,30],[28,32],
  [25,31],[26,32],[0,11],[0,12],[5,11],[6,12]
];

function drawSkeleton(ctx, W, H, lm, jointScores) {
  if (!showSkeleton) return;
  ctx.lineWidth = 3; ctx.lineCap = 'round';

  for (const [a,b] of CONNECTIONS) {
    const la=lm[a], lb=lm[b];
    if (!la||!lb||la.visibility<0.4) continue;
    const scoreA = jointScores[Object.keys(jointScores)[0]]||80;
    ctx.strokeStyle = 'rgba(0,255,136,0.85)';
    ctx.beginPath();
    ctx.moveTo(la.x*W, la.y*H);
    ctx.lineTo(lb.x*W, lb.y*H);
    ctx.stroke();
  }

  for (let i=0;i<lm.length;i++) {
    const l=lm[i];
    if (!l||l.visibility<0.4) continue;
    ctx.fillStyle='#00ff88';
    ctx.strokeStyle='rgba(255,255,255,0.9)';
    ctx.lineWidth=1.5;
    ctx.beginPath();
    ctx.arc(l.x*W, l.y*H, 5, 0, 2*Math.PI);
    ctx.fill(); ctx.stroke();
  }
}

// ═══════════════════════════════════════════════════════════
//  UI UPDATE
// ═══════════════════════════════════════════════════════════
function updateUI(key, result, lm) {
  const ex = EXERCISES[key];

  // Exercise name tags
  document.getElementById('pose-name-tag').textContent = ex?.name || 'Detecting...';
  const st = result.status;
  const stTag = document.getElementById('form-status-tag');
  stTag.textContent = st?.toUpperCase() || 'ANALYZING';
  stTag.className = 'hud-tag ' + (st==='correct'?'green':st==='minor'?'yellow':'red');

  // Score ring
  const score = result.score || 0;
  document.getElementById('score-number').textContent = score;
  const circumference = 283;
  const offset = circumference - (score/100)*circumference;
  const ring = document.getElementById('score-ring-fill');
  ring.style.strokeDashoffset = offset;
  ring.style.stroke = score>=85?'#3D7A52':score>=65?'#C9963A':'#E8604C';

  // Status pill
  const pill = document.getElementById('status-pill');
  pill.textContent = st==='correct'?'✓ GOOD FORM':st==='minor'?'MINOR ISSUES':'NEEDS WORK';
  pill.className = 'status-pill ' + (st==='minor'?'minor':st==='major'?'major':'');

  // Reps
  document.getElementById('rep-num').textContent = repCount;

  // Corrections
  const list = document.getElementById('correction-list');
  list.innerHTML = '';
  if (result.corrections?.length > 0) {
    for (const [,, msg] of result.corrections) {
      const div = document.createElement('div');
      div.className = 'correction-item';
      div.innerHTML = `<span class="correction-dot">●</span><span>${msg}</span>`;
      list.appendChild(div);
    }
  } else if (lm) {
    list.innerHTML = '<div class="good-form">✓ Excellent form! Keep it up.</div>';
  } else {
    list.innerHTML = '<div class="good-form">Position yourself in frame</div>';
  }

  // Joint scores
  const jList = document.getElementById('joints-list');
  jList.innerHTML = '';
  const topJoints = Object.entries(result.jointScores||{}).slice(0,6);
  for (const [name, score] of topJoints) {
    const color = score>=85?'#3D7A52':score>=65?'#C9963A':'#E8604C';
    const label = name.replace(/_/g,' ').replace(/\b\w/g,c=>c.toUpperCase());
    jList.innerHTML += `
      <div class="joint-row">
        <div class="joint-row-top">
          <span class="joint-name">${label}</span>
          <span class="joint-val" style="color:${color}">${Math.round(score)}</span>
        </div>
        <div class="joint-bar-bg">
          <div class="joint-bar-fill" style="width:${score}%;background:${color}"></div>
        </div>
      </div>`;
  }

  // FPS
  frames++;
  const now = Date.now();
  if (now-fpsTimer > 1000) {
    document.getElementById('fps-display').textContent = `${Math.round(frames*1000/(now-fpsTimer))} fps`;
    frames=0; fpsTimer=now;
  }
}

// ═══════════════════════════════════════════════════════════
//  MEDIAPIPE SETUP
// ═══════════════════════════════════════════════════════════
async function initMediaPipe() {
  setLoadMsg('Loading AI model...');
  const vision = await import('https://cdn.jsdelivr.net/npm/@mediapipe/tasks-vision@0.10.14/vision_bundle.mjs');
  const resolver = await vision.FilesetResolver.forVisionTasks(
    'https://cdn.jsdelivr.net/npm/@mediapipe/tasks-vision@0.10.14/wasm'
  );
  poseLandmarker = await vision.PoseLandmarker.createFromOptions(resolver, {
    baseOptions:{
      modelAssetPath:'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task',
      delegate:'GPU'
    },
    runningMode:'VIDEO',
    numPoses:1,
    minPoseDetectionConfidence: detectionConf/100,
    minPosePresenceConfidence:  detectionConf/100,
    minTrackingConfidence:      detectionConf/100,
    outputSegmentationMasks:false
  });
}

// ═══════════════════════════════════════════════════════════
//  CAMERA
// ═══════════════════════════════════════════════════════════
const video = document.getElementById('video');
const canvas = document.getElementById('overlay-canvas');
const ctx = canvas.getContext('2d');
const noMsg = document.getElementById('no-pose-msg');

async function startCamera() {
  setLoadMsg('Starting camera...');
  if (video.srcObject) video.srcObject.getTracks().forEach(t=>t.stop());
  try {
    const stream = await navigator.mediaDevices.getUserMedia({
      video:{facingMode, width:{ideal:1280}, height:{ideal:720}}
    });
    video.srcObject = stream;
    const flip = facingMode==='user'?'scaleX(-1)':'scaleX(1)';
    if (mirrorMode) { video.style.transform=flip; canvas.style.transform=flip; }
    await new Promise(r => { video.onloadedmetadata = () => { canvas.width=video.videoWidth; canvas.height=video.videoHeight; r(); }; });
  } catch(e) {
    setLoadMsg('⚠ Camera error: '+e.message);
    throw e;
  }
}

// ═══════════════════════════════════════════════════════════
//  MAIN RENDER LOOP
// ═══════════════════════════════════════════════════════════
function renderLoop() {
  requestAnimationFrame(renderLoop);
  if (paused || !poseLandmarker || video.readyState<2) return;
  if (video.currentTime === lastVideoTime) return;
  lastVideoTime = video.currentTime;

  let startMs = performance.now();
  if (startMs <= poseTimestamp) startMs = poseTimestamp+1;
  poseTimestamp = startMs;

  try {
    const results = poseLandmarker.detectForVideo(video, startMs);
    ctx.clearRect(0, 0, canvas.width, canvas.height);

    if (results.landmarks?.length > 0) {
      noMsg.style.display='none';
      const raw = results.landmarks[0];
      const lm = raw.map((l,i)=>({x:l.x,y:l.y,z:l.z,visibility:l.visibility??0.9}));

      const angles = getAllAngles(lm);

      // Classify
      if (autoDetect) {
        const detected = classifyPose(angles);
        if (detected !== exerciseKey) {
          exerciseKey = detected;
          repCount=0; repState='idle';
          document.getElementById('exercise-select').value = exerciseKey;
        }
      }

      const result = evaluateForm(angles, exerciseKey);
      updateReps(exerciseKey, angles);
      drawSkeleton(ctx, canvas.width, canvas.height, lm, result.jointScores);
      updateUI(exerciseKey, result, lm);

    } else {
      noMsg.style.display='flex';
      updateUI(exerciseKey, {score:0,corrections:[],status:'unknown',jointScores:{}}, null);
    }
  } catch(e) { console.error(e); }
}

// ═══════════════════════════════════════════════════════════
//  CONTROLS (exposed to window for onclick)
// ═══════════════════════════════════════════════════════════
window.onExerciseChange = function() {
  const sel = document.getElementById('exercise-select').value;
  if (sel) { exerciseKey=sel; autoDetect=false; setMode('manual'); }
  else { autoDetect=true; setMode('auto'); }
  repCount=0; repState='idle';
};

window.setMode = function(mode) {
  autoDetect = mode==='auto';
  document.getElementById('btn-auto').classList.toggle('active', autoDetect);
  document.getElementById('btn-manual').classList.toggle('active', !autoDetect);
};

window.resetReps = function() { repCount=0; repState='idle'; document.getElementById('rep-num').textContent='0'; };

window.flipCamera = async function() {
  facingMode = facingMode==='user'?'environment':'user';
  await startCamera();
};

window.toggleMirror = function() {
  mirrorMode=!mirrorMode;
  const btn=document.getElementById('btn-mirror');
  btn.classList.toggle('active',mirrorMode);
  const t = mirrorMode&&facingMode==='user'?'scaleX(-1)':'scaleX(1)';
  video.style.transform=t; canvas.style.transform=t;
};

window.toggleSkeleton = function() {
  showSkeleton=!showSkeleton;
  document.getElementById('btn-skeleton').classList.toggle('active',showSkeleton);
};

window.togglePause = function() {
  paused=!paused;
  const btn=document.getElementById('btn-pause');
  btn.classList.toggle('active',paused);
  btn.innerHTML = paused
    ? '<svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><polygon points="5 3 19 12 5 21 5 3"/></svg> Resume'
    : '<svg width="14" height="14" viewBox="0 0 24 24" fill="none" stroke="currentColor" stroke-width="2"><rect x="6" y="4" width="4" height="16"/><rect x="14" y="4" width="4" height="16"/></svg> Pause';
};

window.applyFilters = function() {
  const b=document.getElementById('sl-brightness').value;
  const c=document.getElementById('sl-contrast').value;
  const s=document.getElementById('sl-saturation').value;
  video.style.filter=`brightness(${b}%) contrast(${c}%) saturate(${s}%)`;
  document.getElementById('val-brightness').textContent=b+'%';
  document.getElementById('val-contrast').textContent=c+'%';
  document.getElementById('val-saturation').textContent=s+'%';
};

window.updateSensitivity = function() {
  const v=document.getElementById('sl-sensitivity').value;
  document.getElementById('val-sensitivity').textContent=v+'%';
  detectionConf = v/100;
};

// ═══════════════════════════════════════════════════════════
//  BOOT
// ═══════════════════════════════════════════════════════════
function setLoadMsg(msg) { document.getElementById('load-msg').textContent=msg; }

async function boot() {
  document.getElementById('scan-line').style.display='block';
  await initMediaPipe();
  await startCamera();
  document.getElementById('loading-overlay').style.opacity='0';
  setTimeout(()=>{ document.getElementById('loading-overlay').style.display='none'; },500);
  document.getElementById('scan-line').style.display='none';
  renderLoop();
}

boot().catch(e=>{ setLoadMsg('❌ Error: '+e.message); console.error(e); });
</script>
</body>
</html>"""

with open("/content/pose_app/index.html", "w") as f:
    f.write(HTML)

print("✅ index.html written —", len(HTML), "chars")

✅ index.html written — 43360 chars


In [ ]:

SERVER_CODE = '''
from flask import Flask, send_from_directory
from flask_cors import CORS
import os

app = Flask(__name__, static_folder="/content/pose_app")
CORS(app)

@app.route("/")
def index():
    return send_from_directory("/content/pose_app", "index.html")

@app.route("/<path:path>")
def static_files(path):
    return send_from_directory("/content/pose_app", path)

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)
'''

with open("/content/pose_server.py", "w") as f:
    f.write(SERVER_CODE)

print("✅ Flask server written.")

✅ Flask server written.


###**Need your NGROK Auth token from your account in NGROK website.**

In [ ]:
from pyngrok import ngrok, conf

NGROK_TOKEN = " "     #Paste your AUTH Token here

ngrok.set_auth_token(NGROK_TOKEN)
print("✅ ngrok auth token set.")

✅ ngrok auth token set.


In [ ]:
import subprocess, time, threading
from pyngrok import ngrok

# Kill any existing ngrok tunnels
ngrok.kill()

def run_flask():
    subprocess.run(["python", "/content/pose_server.py"])

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

time.sleep(2)

tunnel = ngrok.connect(5000)
public_url = tunnel.public_url

print("\n" + "="*60)
print("  🎉 POSE CORRECTION APP IS LIVE!")
print("="*60)
print(f"\n  👉  Open this link in your browser:\n")
print(f"      {public_url}\n")
print("="*60)
print("\n  📋  Instructions:")
print("  1. Click the link above")
print("  2. Allow camera access when browser asks")
print("  3. Stand in front of camera (full body visible)")
print("  4. Select an exercise or use Auto-Detect")
print("  5. Check your form score live!")
print("\n  ⚠  Keep this Colab tab open while using the app.")
print("="*60)


  🎉 POSE CORRECTION APP IS LIVE!

  👉  Open this link in your browser:

      https://1b8d-136-107-90-127.ngrok-free.app


  📋  Instructions:
  1. Click the link above
  2. Allow camera access when browser asks
  3. Stand in front of camera (full body visible)
  4. Select an exercise or use Auto-Detect
  5. Check your form score live!

  ⚠  Keep this Colab tab open while using the app.
